In [62]:
# ==========================================
# STEP 1. 라이브러리
# ==========================================

import os
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [63]:
# ==========================================
# STEP 2. ZIP 압축 해제
# ==========================================

zip_path = "/content/fraud_full_features.zip"
extract_path = "/content/fraud_full_features"

os.makedirs(
    extract_path,
    exist_ok=True
)

with zipfile.ZipFile(
    zip_path,
    "r"
) as zip_ref:

    zip_ref.extractall(
        extract_path
    )

print("압축 해제 완료")

print("\n압축 해제된 파일:")
print(
    os.listdir(
        extract_path
    )
)

압축 해제 완료

압축 해제된 파일:
['fraud_full_features.csv']


In [64]:
# ==========================================
# STEP 3. 데이터 로드
# ==========================================

df = pd.read_csv(
    "/content/fraud_full_features/fraud_full_features.csv"
)

print("데이터 로드 완료")

print(
    "전체 데이터:",
    df.shape
)

print(
    "전체 컬럼:",
    len(df.columns)
)

데이터 로드 완료
전체 데이터: (1296675, 31)
전체 컬럼: 31


In [65]:
# ==========================================
# STEP 4. 시간순 정렬
# ==========================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

print("전체 기간")
print(
    df["trans_date_trans_time"].min(),
    "~",
    df["trans_date_trans_time"].max()
)

print(
    "\n전체 거래:",
    len(df)
)

print(
    "전체 이상거래:",
    int(df["is_fraud"].sum())
)

print(
    "전체 이상거래율:",
    f"{df['is_fraud'].mean() * 100:.4f}%"
)

전체 기간
2019-01-01 00:00:18 ~ 2020-06-21 12:13:37

전체 거래: 1296675
전체 이상거래: 7506
전체 이상거래율: 0.5789%


In [66]:
# ==========================================
# STEP 5. Feature Set 정의
# ==========================================

set1 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min"
]


set2 = set1 + [
    "merchant_change_count"
]


set3 = set1 + [
    "high_speed"
]


set4 = [
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "category",
    "amt",
    "trans_hour",
    "age"
]


set5 = [
    "category",
    "amt",
    "is_online",
    "recent_24h_high_amt_count",
    "category_recent_fraud_rate",
    "speed_2",
    "customer_mean_amt",
    "customer_std_amt",
    "amt_ratio_to_mean",
    "amt_zscore_card",
    "customer_transaction_count",
    "trans_hour",
    "age",
    "rolling_sum_amt_1h",
    "prior_normal_median_amt",
    "amt_to_prior_median_ratio",
    "risk_time_22_04",
    "interact_repeat_category",
    "has_prior_normal_transaction"
]


feature_sets = {
    "Set 1": set1,
    "Set 2": set2,
    "Set 3": set3,
    "Set 4": set4,
    "Set 5": set5
}


for name, features in feature_sets.items():

    print(
        name,
        ":",
        len(features),
        "개"
    )

Set 1 : 10 개
Set 2 : 11 개
Set 3 : 11 개
Set 4 : 6 개
Set 5 : 19 개


In [67]:
# ==========================================
# STEP 6. 변수 존재 확인
# ==========================================

for set_name, features in feature_sets.items():

    missing = [
        x
        for x in features
        if x not in df.columns
    ]

    if not missing:

        print(
            f"✅ {set_name}: "
            f"모든 변수 존재"
        )

    else:

        print(
            f"❌ {set_name}: "
            f"{missing}"
        )

✅ Set 1: 모든 변수 존재
✅ Set 2: 모든 변수 존재
✅ Set 3: 모든 변수 존재
✅ Set 4: 모든 변수 존재
✅ Set 5: 모든 변수 존재


In [68]:
# ==========================================
# STEP 7. 결측치 확인
# ==========================================

all_features = sorted(
    set(
        feature
        for features in feature_sets.values()
        for feature in features
    )
)

missing_count = (
    df[all_features]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    missing_count[
        missing_count > 0
    ]
)

,0
amt_to_prior_median_ratio,1649
prior_normal_median_amt,1649


In [80]:
# ==========================================
# STEP 7-A. 시간순 80:20 분할
# ==========================================

split_idx = int(len(df) * 0.8)

train_df_80 = df.iloc[:split_idx].copy()
val_df_20 = df.iloc[split_idx:].copy()

print("===== 시간순 80:20 =====")

print("Train shape :", train_df_80.shape)
print("Valid shape :", val_df_20.shape)

print("\nTrain 기간")
print(
    train_df_80["trans_date_trans_time"].min(),
    "~",
    train_df_80["trans_date_trans_time"].max()
)

print("\nValidation 기간")
print(
    val_df_20["trans_date_trans_time"].min(),
    "~",
    val_df_20["trans_date_trans_time"].max()
)

===== 시간순 80:20 =====
Train shape : (1037340, 31)
Valid shape : (259335, 31)

Train 기간
2019-01-01 00:00:18 ~ 2020-03-06 07:15:17

Validation 기간
2020-03-06 07:16:43 ~ 2020-06-21 12:13:37


In [81]:
# ==========================================
# STEP 7-B. 80% Train의 class weight 계산
# ==========================================

y_train_80 = train_df_80["is_fraud"]

normal_80 = (y_train_80 == 0).sum()
fraud_80 = (y_train_80 == 1).sum()

scale_pos_weight_80 = normal_80 / fraud_80

class_weight_80 = {
    0: 1.0,
    1: scale_pos_weight_80
}

print("Normal :", normal_80)
print("Fraud  :", fraud_80)
print("Positive Class Weight :", scale_pos_weight_80)

Normal : 1031372
Fraud  : 5968
Positive Class Weight : 172.8170241286863


In [82]:
# ==========================================
# STEP 7-C. Logistic Pipeline
# class_weight를 외부에서 받도록 수정
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


CATEGORICAL_COLUMNS = ["category"]


def make_logistic_pipeline(
    features,
    class_weight,
    C=1.0,
    penalty="l2",
    solver="liblinear"
):

    categorical_features = [
        col
        for col in features
        if col in CATEGORICAL_COLUMNS
    ]

    numeric_features = [
        col
        for col in features
        if col not in CATEGORICAL_COLUMNS
    ]


    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ])


    logistic = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        class_weight=class_weight,
        max_iter=100,
        random_state=42
    )


    return Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            logistic
        )
    ])

In [83]:
# ==========================================
# STEP 7-D. 평가 함수
# ==========================================

def evaluate_binary_model(
    y_true,
    prob,
    threshold=0.90
):

    pred = (
        prob >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    ).ravel()


    return {
        "PR-AUC":
            average_precision_score(
                y_true,
                prob
            ),

        "ROC-AUC":
            roc_auc_score(
                y_true,
                prob
            ),

        "Accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "Precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "FP": fp,
        "FN": fn,
        "TP": tp,
        "TN": tn
    }

In [84]:
# ==========================================
# STEP 7-E. 80:20 - 5개 변수조합 비교
# ==========================================

results_80 = []


for set_name, features in feature_sets.items():

    print("\n" + "=" * 70)
    print(f"80:20 | {set_name}")
    print("=" * 70)


    X_train = train_df_80[features]
    y_train = train_df_80["is_fraud"]

    X_val = val_df_20[features]
    y_val = val_df_20["is_fraud"]


    model = make_logistic_pipeline(
        features=features,
        class_weight=class_weight_80,
        C=1.0,
        penalty="l2",
        solver="liblinear"
    )


    model.fit(
        X_train,
        y_train
    )


    val_prob = model.predict_proba(
        X_val
    )[:, 1]


    metrics = evaluate_binary_model(
        y_true=y_val,
        prob=val_prob,
        threshold=THRESHOLD
    )


    results_80.append({

        "Feature Set":
            set_name,

        "N Features":
            len(features),

        "Threshold":
            THRESHOLD,

        "Class Weight":
            scale_pos_weight_80,

        **metrics
    })


    print(
        f"PR-AUC     : {metrics['PR-AUC']:.6f}"
    )

    print(
        f"Precision  : {metrics['Precision']:.6f}"
    )

    print(
        f"Recall     : {metrics['Recall']:.6f}"
    )

    print(
        f"F1         : {metrics['F1']:.6f}"
    )

    print(
        f"FP={metrics['FP']:,} | "
        f"FN={metrics['FN']:,}"
    )


80:20 | Set 1
PR-AUC     : 0.447992
Precision  : 0.358591
Recall     : 0.747724
F1         : 0.484721
FP=2,057 | FN=388

80:20 | Set 2
PR-AUC     : 0.450318
Precision  : 0.363295
Recall     : 0.754226
F1         : 0.490383
FP=2,033 | FN=378

80:20 | Set 3
PR-AUC     : 0.447357
Precision  : 0.358591
Recall     : 0.747724
F1         : 0.484721
FP=2,057 | FN=388

80:20 | Set 4
PR-AUC     : 0.417695
Precision  : 0.320513
Recall     : 0.731469
F1         : 0.445721
FP=2,385 | FN=413

80:20 | Set 5
PR-AUC     : 0.545277
Precision  : 0.508254
Recall     : 0.760728
F1         : 0.609375
FP=1,132 | FN=368


In [85]:
# ==========================================
# STEP 7-F. 80:20 결과 정렬
# ==========================================

results_80_df = pd.DataFrame(
    results_80
)


results_80_df = (
    results_80_df
    .sort_values(
        by=[
            "PR-AUC",
            "F1",
            "Recall"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


results_80_df.insert(
    0,
    "Rank",
    range(
        1,
        len(results_80_df) + 1
    )
)


display(
    results_80_df[
        [
            "Rank",
            "Feature Set",
            "PR-AUC",
            "Precision",
            "Recall",
            "F1",
            "FP",
            "FN",
            "Class Weight"
        ]
    ]
)

,Rank,Feature Set,PR-AUC,Precision,Recall,F1,FP,FN,Class Weight
0,1,Set 5,0.545277,0.508254,0.760728,0.609375,1132,368,172.817024
1,2,Set 2,0.450318,0.363295,0.754226,0.490383,2033,378,172.817024
2,3,Set 1,0.447992,0.358591,0.747724,0.484721,2057,388,172.817024
3,4,Set 3,0.447357,0.358591,0.747724,0.484721,2057,388,172.817024
4,5,Set 4,0.417695,0.320513,0.731469,0.445721,2385,413,172.817024


In [86]:
# ==========================================
# STEP 8. 공통 실험 설정
# ==========================================

RANDOM_STATE = 42
THRESHOLD = 0.90

# 팀 공통 불균형 처리 가중치
normal = (y_train == 0).sum()
fraud = (y_train == 1).sum()

scale_pos_weight = normal / fraud

class_weight = {
    0: 1.0,
    1: scale_pos_weight
}

# 학습 중단 조건
MAX_ITER = 100


print("Threshold           :", THRESHOLD)
print("Positive Class Weight:", POS_WEIGHT)
print("Max Iteration       :", MAX_ITER)
print("Random State        :", RANDOM_STATE)

Threshold           : 0.9
Positive Class Weight: 171.746045
Max Iteration       : 100
Random State        : 42


In [93]:
# ==========================================
# STEP 9. Logistic Regression Pipeline
# ==========================================

CATEGORICAL_COLUMNS = [
    "category"
]


def make_logistic_pipeline(
    features,
    class_weight,
    C=1.0,
    penalty="l2",
    solver="liblinear"
):

    # ------------------------------------------
    # 1. 범주형 / 수치형 변수 구분
    # ------------------------------------------

    categorical_features = [
        col
        for col in features
        if col in CATEGORICAL_COLUMNS
    ]

    numeric_features = [
        col
        for col in features
        if col not in CATEGORICAL_COLUMNS
    ]


    # ------------------------------------------
    # 2. 수치형 변수 전처리
    # 결측치 중앙값 대체 → StandardScaler
    # ------------------------------------------

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    # ------------------------------------------
    # 3. 범주형 변수 전처리
    # 결측치 최빈값 대체 → One-Hot Encoding
    # ------------------------------------------

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ])


    # ------------------------------------------
    # 4. 수치형 + 범주형 전처리 결합
    # ------------------------------------------

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ])


    # ------------------------------------------
    # 5. Logistic Regression
    # class_weight는 각 Train 구간에서
    # normal / fraud로 계산해서 외부에서 전달
    # ------------------------------------------

    logistic = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        class_weight=class_weight,
        max_iter=100,
        random_state=42
    )


    # ------------------------------------------
    # 6. 최종 Pipeline
    # ------------------------------------------

    model = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            logistic
        )
    ])

    return model

In [88]:
# ==========================================
# STEP 10. 평가 함수
# ==========================================

def evaluate_model(
    y_true,
    probability,
    threshold=THRESHOLD
):

    prediction = (
        probability >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1]
    ).ravel()

    return {

        "PR-AUC":
            average_precision_score(
                y_true,
                probability
            ),

        "ROC-AUC":
            roc_auc_score(
                y_true,
                probability
            ),

        "Accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "Precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "FP": fp,
        "FN": fn,
        "TP": tp,
        "TN": tn
    }

In [89]:
# ==========================================
# STEP 11. 시간순 80:20
# ==========================================

split_index = int(
    len(df) * 0.8
)

train_df_80 = (
    df
    .iloc[:split_index]
    .copy()
)

val_df_20 = (
    df
    .iloc[split_index:]
    .copy()
)


print("===== Train 80% =====")

print(
    "거래:",
    len(train_df_80)
)

print(
    "Fraud:",
    int(
        train_df_80[
            "is_fraud"
        ].sum()
    )
)

print(
    "Fraud Rate:",
    f"{train_df_80['is_fraud'].mean()*100:.4f}%"
)


print("\n===== Validation 20% =====")

print(
    "거래:",
    len(val_df_20)
)

print(
    "Fraud:",
    int(
        val_df_20[
            "is_fraud"
        ].sum()
    )
)

print(
    "Fraud Rate:",
    f"{val_df_20['is_fraud'].mean()*100:.4f}%"
)

===== Train 80% =====
거래: 1037340
Fraud: 5968
Fraud Rate: 0.5753%

===== Validation 20% =====
거래: 259335
Fraud: 1538
Fraud Rate: 0.5931%


In [90]:
# ==========================================
# STEP 12. 80:20 Feature Set 비교
# ==========================================

results_80 = []


for set_name, features in feature_sets.items():

    print("\n" + "=" * 70)
    print(set_name)
    print("=" * 70)

    X_train = train_df_80[
        features
    ]

    y_train = train_df_80[
        "is_fraud"
    ]

    X_val = val_df_20[
        features
    ]

    y_val = val_df_20[
        "is_fraud"
    ]


    model = make_logistic_pipeline(
        features=features,
        C=1.0,
        penalty="l2",
        solver="liblinear"
    )


    model.fit(
        X_train,
        y_train
    )


    val_prob = model.predict_proba(
        X_val
    )[:, 1]


    metrics = evaluate_model(
        y_val,
        val_prob
    )


    results_80.append({

        "Feature Set":
            set_name,

        "N Features":
            len(features),

        **metrics
    })


results_80_df = pd.DataFrame(
    results_80
)


results_80_df = (
    results_80_df
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    results_80_df
)


Set 1

Set 2

Set 3

Set 4

Set 5


,Feature Set,N Features,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1,FP,FN,TP,TN
0,Set 5,19,0.545469,0.985233,0.994243,0.509804,0.760728,0.610488,1125,368,1170,256672
1,Set 2,11,0.450422,0.971843,0.990719,0.363750,0.754226,0.490798,2029,378,1160,255768
2,Set 1,10,0.448107,0.971711,0.990611,0.359624,0.747074,0.485527,2046,389,1149,255751
3,Set 3,11,0.447474,0.971694,0.990630,0.360276,0.747724,0.486258,2042,388,1150,255755
4,Set 4,6,0.417830,0.970282,0.989323,0.323183,0.731469,0.448296,2356,413,1125,255441


In [74]:
# ==========================================
# STEP 13. 시간순 데이터 행 기준 6등분
# ==========================================

segments = np.array_split(
    df,
    6
)

segment_summary = []


for i, segment in enumerate(
    segments,
    start=1
):

    segment_summary.append({

        "Segment":
            i,

        "Transactions":
            len(segment),

        "Fraud":
            int(
                segment[
                    "is_fraud"
                ].sum()
            ),

        "Fraud Rate (%)":
            segment[
                "is_fraud"
            ].mean() * 100,

        "Start":
            segment[
                "trans_date_trans_time"
            ].min(),

        "End":
            segment[
                "trans_date_trans_time"
            ].max()
    })


segment_summary_df = pd.DataFrame(
    segment_summary
)


display(
    segment_summary_df
)

,Segment,Transactions,Fraud,Fraud Rate (%),Start,End
0,1,216113,1742,0.806060,2019-01-01 00:00:18,2019-04-20 12:53:31
1,2,216113,1054,0.487708,2019-04-20 12:53:44,2019-07-12 23:49:08
2,3,216113,1031,0.477065,2019-07-12 23:50:14,2019-10-03 07:36:02
3,4,216112,1091,0.504831,2019-10-03 07:36:54,2019-12-18 17:07:05
4,5,216112,1363,0.630691,2019-12-18 17:07:56,2020-03-24 15:14:30
5,6,216112,1225,0.566836,2020-03-24 15:15:08,2020-06-21 12:13:37


In [75]:
# ==========================================
# STEP 14. 3-Fold Expanding Window
# ==========================================

folds = {

    "Fold 1": {
        "train_segments": [0, 1, 2],
        "val_segment": 3
    },

    "Fold 2": {
        "train_segments": [0, 1, 2, 3],
        "val_segment": 4
    },

    "Fold 3": {
        "train_segments": [0, 1, 2, 3, 4],
        "val_segment": 5
    }
}


def get_fold_data(
    segments,
    fold_info
):

    fold_train = pd.concat(
        [
            segments[i]
            for i in fold_info[
                "train_segments"
            ]
        ],
        axis=0
    ).copy()

    fold_val = (
        segments[
            fold_info[
                "val_segment"
            ]
        ]
        .copy()
    )

    return (
        fold_train,
        fold_val
    )

In [76]:
# ==========================================
# STEP 15. Fold 확인
# ==========================================

fold_summary = []


for fold_name, fold_info in folds.items():

    fold_train, fold_val = (
        get_fold_data(
            segments,
            fold_info
        )
    )


    fold_summary.append({

        "Fold":
            fold_name,

        "Train N":
            len(fold_train),

        "Validation N":
            len(fold_val),

        "Train Fraud":
            int(
                fold_train[
                    "is_fraud"
                ].sum()
            ),

        "Validation Fraud":
            int(
                fold_val[
                    "is_fraud"
                ].sum()
            ),

        "Validation Fraud Rate (%)":
            fold_val[
                "is_fraud"
            ].mean() * 100,

        "Train End":
            fold_train[
                "trans_date_trans_time"
            ].max(),

        "Validation Start":
            fold_val[
                "trans_date_trans_time"
            ].min()
    })


fold_summary_df = pd.DataFrame(
    fold_summary
)


display(
    fold_summary_df
)

,Fold,Train N,Validation N,Train Fraud,Validation Fraud,Validation Fraud Rate (%),Train End,Validation Start
0,Fold 1,648339,216112,3827,1091,0.504831,2019-10-03 07:36:02,2019-10-03 07:36:54
1,Fold 2,864451,216112,4918,1363,0.630691,2019-12-18 17:07:05,2019-12-18 17:07:56
2,Fold 3,1080563,216112,6281,1225,0.566836,2020-03-24 15:14:30,2020-03-24 15:15:08


In [94]:
# ==========================================
# STEP 16. 튜닝 전 3-Fold 기본 성능 비교
# ==========================================

baseline_cv_results = []


for set_name, features in feature_sets.items():

    print("\n" + "=" * 80)
    print(f"BASELINE 3-FOLD | {set_name}")
    print("=" * 80)

    fold_scores = []


    for fold_name, fold_info in folds.items():

        # ------------------------------------------
        # Fold 데이터
        # ------------------------------------------
        fold_train, fold_val = get_fold_data(
            segments,
            fold_info
        )

        X_train = fold_train[features]
        y_train = fold_train["is_fraud"]

        X_val = fold_val[features]
        y_val = fold_val["is_fraud"]


        # ------------------------------------------
        # ★ 현재 Fold의 Train에서만 가중치 계산
        # ------------------------------------------
        normal = (y_train == 0).sum()
        fraud = (y_train == 1).sum()

        scale_pos_weight = normal / fraud

        class_weight = {
            0: 1.0,
            1: scale_pos_weight
        }


        # ------------------------------------------
        # 기본 Logistic Regression
        # ------------------------------------------
        model = make_logistic_pipeline(
            features=features,
            class_weight=class_weight,
            C=1.0,
            penalty="l2",
            solver="liblinear"
        )


        model.fit(
            X_train,
            y_train
        )


        # ------------------------------------------
        # Validation 예측확률
        # ------------------------------------------
        val_prob = model.predict_proba(
            X_val
        )[:, 1]


        # ------------------------------------------
        # PR-AUC
        # ------------------------------------------
        pr_auc = average_precision_score(
            y_val,
            val_prob
        )

        fold_scores.append(
            pr_auc
        )


        print(
            f"{fold_name} | "
            f"Weight={scale_pos_weight:.6f} | "
            f"PR-AUC={pr_auc:.6f}"
        )


    # ------------------------------------------
    # Set별 결과
    # ------------------------------------------
    baseline_cv_results.append({

        "Feature Set":
            set_name,

        "Fold 1 PR-AUC":
            fold_scores[0],

        "Fold 2 PR-AUC":
            fold_scores[1],

        "Fold 3 PR-AUC":
            fold_scores[2],

        "Mean PR-AUC":
            np.mean(fold_scores),

        "Std PR-AUC":
            np.std(fold_scores)
    })


baseline_cv_df = pd.DataFrame(
    baseline_cv_results
)


baseline_cv_df = (
    baseline_cv_df
    .sort_values(
        [
            "Mean PR-AUC",
            "Std PR-AUC"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


baseline_cv_df.insert(
    0,
    "Rank",
    range(
        1,
        len(baseline_cv_df) + 1
    )
)


display(
    baseline_cv_df
)


BASELINE 3-FOLD | Set 1
Fold 1 | Weight=168.411811 | PR-AUC=0.417594
Fold 2 | Weight=174.772875 | PR-AUC=0.467473
Fold 3 | Weight=171.036778 | PR-AUC=0.431729

BASELINE 3-FOLD | Set 2
Fold 1 | Weight=168.411811 | PR-AUC=0.419781
Fold 2 | Weight=174.772875 | PR-AUC=0.468706
Fold 3 | Weight=171.036778 | PR-AUC=0.434036

BASELINE 3-FOLD | Set 3
Fold 1 | Weight=168.411811 | PR-AUC=0.416150
Fold 2 | Weight=174.772875 | PR-AUC=0.466320
Fold 3 | Weight=171.036778 | PR-AUC=0.431234

BASELINE 3-FOLD | Set 4
Fold 1 | Weight=168.411811 | PR-AUC=0.388559
Fold 2 | Weight=174.772875 | PR-AUC=0.437878
Fold 3 | Weight=171.036778 | PR-AUC=0.404770

BASELINE 3-FOLD | Set 5
Fold 1 | Weight=168.411811 | PR-AUC=0.513423
Fold 2 | Weight=174.772875 | PR-AUC=0.556721
Fold 3 | Weight=171.036778 | PR-AUC=0.529711


,Rank,Feature Set,Fold 1 PR-AUC,Fold 2 PR-AUC,Fold 3 PR-AUC,Mean PR-AUC,Std PR-AUC
0,1,Set 5,0.513423,0.556721,0.529711,0.533285,0.017856
1,2,Set 2,0.419781,0.468706,0.434036,0.440841,0.020545
2,3,Set 1,0.417594,0.467473,0.431729,0.438932,0.020990
3,4,Set 3,0.416150,0.466320,0.431234,0.437901,0.021018
4,5,Set 4,0.388559,0.437878,0.404770,0.410402,0.020525


In [95]:
# ==========================================
# STEP 17. 튜닝 대상 Feature Set 선정
# ==========================================

# Baseline 3-Fold 결과에서 일부 조합 간
# Mean PR-AUC 차이가 매우 작게 나타남.
#
# 하이퍼파라미터 튜닝 이후 순위가 달라질 수 있으므로
# Set 1~5 전체를 튜닝 대상으로 유지한다.

selected_sets = [
    "Set 1",
    "Set 2",
    "Set 3",
    "Set 4",
    "Set 5"
]


print("=" * 70)
print("하이퍼파라미터 튜닝 대상 Feature Set")
print("=" * 70)


# STEP 16의 Baseline 성능도 함께 출력
for set_name in selected_sets:

    row = baseline_cv_df.loc[
        baseline_cv_df["Feature Set"] == set_name
    ].iloc[0]

    print(
        f"{set_name} | "
        f"Mean PR-AUC = {row['Mean PR-AUC']:.6f} | "
        f"Std PR-AUC = {row['Std PR-AUC']:.6f}"
    )


print("\n최종 튜닝 대상:")
print(selected_sets)

print(
    f"\n총 {len(selected_sets)}개 Feature Set을 "
    "하이퍼파라미터 튜닝합니다."
)

하이퍼파라미터 튜닝 대상 Feature Set
Set 1 | Mean PR-AUC = 0.438932 | Std PR-AUC = 0.020990
Set 2 | Mean PR-AUC = 0.440841 | Std PR-AUC = 0.020545
Set 3 | Mean PR-AUC = 0.437901 | Std PR-AUC = 0.021018
Set 4 | Mean PR-AUC = 0.410402 | Std PR-AUC = 0.020525
Set 5 | Mean PR-AUC = 0.533285 | Std PR-AUC = 0.017856

최종 튜닝 대상:
['Set 1', 'Set 2', 'Set 3', 'Set 4', 'Set 5']

총 5개 Feature Set을 하이퍼파라미터 튜닝합니다.


In [96]:
# ==========================================
# STEP 18. Logistic Hyperparameter Grid
# ==========================================

param_grid = [

    # L2
    {
        "C": 0.01,
        "penalty": "l2",
        "solver": "liblinear"
    },

    {
        "C": 0.1,
        "penalty": "l2",
        "solver": "liblinear"
    },

    {
        "C": 1.0,
        "penalty": "l2",
        "solver": "liblinear"
    },

    {
        "C": 10.0,
        "penalty": "l2",
        "solver": "liblinear"
    },

    # L1
    {
        "C": 0.01,
        "penalty": "l1",
        "solver": "liblinear"
    },

    {
        "C": 0.1,
        "penalty": "l1",
        "solver": "liblinear"
    },

    {
        "C": 1.0,
        "penalty": "l1",
        "solver": "liblinear"
    },

    {
        "C": 10.0,
        "penalty": "l1",
        "solver": "liblinear"
    }
]


print(
    "하이퍼파라미터 조합:",
    len(param_grid),
    "개"
)

print(
    "튜닝 Feature Set:",
    selected_sets
)

하이퍼파라미터 조합: 8 개
튜닝 Feature Set: ['Set 1', 'Set 2', 'Set 3', 'Set 4', 'Set 5']


In [97]:
# ==========================================
# STEP 19. 선정 Set 3-Fold Hyperparameter Tuning
# ==========================================

tuning_results = []


for set_name in selected_sets:

    features = feature_sets[
        set_name
    ]

    print("\n" + "=" * 80)
    print(f"TUNING | {set_name}")
    print("=" * 80)


    for params in param_grid:

        fold_pr_auc = []


        for fold_name, fold_info in folds.items():

            # --------------------------------------
            # Fold 데이터
            # --------------------------------------
            fold_train, fold_val = get_fold_data(
                segments,
                fold_info
            )

            X_train = fold_train[features]
            y_train = fold_train["is_fraud"]

            X_val = fold_val[features]
            y_val = fold_val["is_fraud"]


            # --------------------------------------
            # ★ Fold별 Train 가중치
            # --------------------------------------
            normal = (y_train == 0).sum()
            fraud = (y_train == 1).sum()

            scale_pos_weight = (
                normal / fraud
            )

            class_weight = {
                0: 1.0,
                1: scale_pos_weight
            }


            # --------------------------------------
            # 모델
            # --------------------------------------
            model = make_logistic_pipeline(
                features=features,
                class_weight=class_weight,
                C=params["C"],
                penalty=params["penalty"],
                solver=params["solver"]
            )


            model.fit(
                X_train,
                y_train
            )


            # --------------------------------------
            # Validation 확률
            # --------------------------------------
            val_prob = model.predict_proba(
                X_val
            )[:, 1]


            pr_auc = average_precision_score(
                y_val,
                val_prob
            )


            fold_pr_auc.append(
                pr_auc
            )


        # ------------------------------------------
        # Hyperparameter 조합별 결과
        # ------------------------------------------
        tuning_results.append({

            "Feature Set":
                set_name,

            "C":
                params["C"],

            "Penalty":
                params["penalty"],

            "Solver":
                params["solver"],

            "Fold 1 PR-AUC":
                fold_pr_auc[0],

            "Fold 2 PR-AUC":
                fold_pr_auc[1],

            "Fold 3 PR-AUC":
                fold_pr_auc[2],

            "Mean PR-AUC":
                np.mean(fold_pr_auc),

            "Std PR-AUC":
                np.std(fold_pr_auc)
        })


tuning_results_df = pd.DataFrame(
    tuning_results
)


display(
    tuning_results_df
)


TUNING | Set 1

TUNING | Set 2

TUNING | Set 3

TUNING | Set 4

TUNING | Set 5


,Feature Set,C,Penalty,Solver,Fold 1 PR-AUC,Fold 2 PR-AUC,Fold 3 PR-AUC,Mean PR-AUC,Std PR-AUC
0,Set 1,0.010000,l2,liblinear,0.422112,0.470722,0.434713,0.442516,0.020597
1,Set 1,0.100000,l2,liblinear,0.418157,0.467865,0.432092,0.439372,0.020936
2,Set 1,1.000000,l2,liblinear,0.417594,0.467473,0.431729,0.438932,0.020990
3,Set 1,10.000000,l2,liblinear,0.417528,0.467437,0.431700,0.438888,0.021000
4,Set 1,0.010000,l1,liblinear,0.421562,0.470259,0.434250,0.442024,0.020627
5,Set 1,0.100000,l1,liblinear,0.417976,0.467758,0.431994,0.439243,0.020960
6,Set 1,1.000000,l1,liblinear,0.417565,0.467467,0.431722,0.438918,0.020998
7,Set 1,10.000000,l1,liblinear,0.417530,0.467435,0.431700,0.438889,0.020998
8,Set 2,0.010000,l2,liblinear,0.424218,0.471942,0.436986,0.444382,0.020173
9,Set 2,0.100000,l2,liblinear,0.420350,0.469067,0.434355,0.441257,0.020479


In [98]:
# ==========================================
# STEP 20. Feature Set별 Best Parameter
# ==========================================

best_params_df = (

    tuning_results_df

    .sort_values(
        [
            "Feature Set",
            "Mean PR-AUC",
            "Std PR-AUC"
        ],

        ascending=[
            True,
            False,
            True
        ]
    )

    .groupby(
        "Feature Set",
        as_index=False
    )

    .first()
)


best_params_df = (
    best_params_df
    .sort_values(
        [
            "Mean PR-AUC",
            "Std PR-AUC"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


display(
    best_params_df
)

,Feature Set,C,Penalty,Solver,Fold 1 PR-AUC,Fold 2 PR-AUC,Fold 3 PR-AUC,Mean PR-AUC,Std PR-AUC
0,Set 5,0.010000,l2,liblinear,0.519429,0.560903,0.533482,0.537938,0.017222
1,Set 2,0.010000,l2,liblinear,0.424218,0.471942,0.436986,0.444382,0.020173
2,Set 1,0.010000,l2,liblinear,0.422112,0.470722,0.434713,0.442516,0.020597
3,Set 3,0.010000,l2,liblinear,0.420746,0.469597,0.434411,0.441584,0.020578
4,Set 4,0.010000,l2,liblinear,0.393398,0.441352,0.408172,0.414307,0.020052


In [99]:
# ==========================================
# STEP 21. Best Model의 OOF Validation 확률 수집
# ==========================================

oof_results = {}


for _, row in best_params_df.iterrows():

    set_name = row[
        "Feature Set"
    ]

    features = feature_sets[
        set_name
    ]

    best_C = row[
        "C"
    ]

    best_penalty = row[
        "Penalty"
    ]

    best_solver = row[
        "Solver"
    ]


    all_y_true = []
    all_y_prob = []

    fold_detail = []


    print("\n" + "=" * 80)
    print(f"OOF 수집 | {set_name}")
    print("=" * 80)


    for fold_name, fold_info in folds.items():

        fold_train, fold_val = get_fold_data(
            segments,
            fold_info
        )


        X_train = fold_train[
            features
        ]

        y_train = fold_train[
            "is_fraud"
        ]

        X_val = fold_val[
            features
        ]

        y_val = fold_val[
            "is_fraud"
        ]


        # --------------------------------------
        # Fold Train에서 가중치 계산
        # --------------------------------------
        normal = (
            y_train == 0
        ).sum()

        fraud = (
            y_train == 1
        ).sum()

        scale_pos_weight = (
            normal / fraud
        )

        class_weight = {
            0: 1.0,
            1: scale_pos_weight
        }


        # --------------------------------------
        # Best Parameter 모델
        # --------------------------------------
        model = make_logistic_pipeline(

            features=features,

            class_weight=class_weight,

            C=best_C,

            penalty=best_penalty,

            solver=best_solver
        )


        model.fit(
            X_train,
            y_train
        )


        val_prob = model.predict_proba(
            X_val
        )[:, 1]


        fold_pr_auc = (
            average_precision_score(
                y_val,
                val_prob
            )
        )


        # --------------------------------------
        # 실제값 + 예측확률 통합
        # --------------------------------------
        all_y_true.extend(
            y_val.to_numpy()
        )

        all_y_prob.extend(
            val_prob
        )


        fold_detail.append({

            "Fold":
                fold_name,

            "Weight":
                scale_pos_weight,

            "PR-AUC":
                fold_pr_auc,

            "N":
                len(y_val),

            "Fraud":
                int(y_val.sum())
        })


        print(
            f"{fold_name} | "
            f"Weight={scale_pos_weight:.6f} | "
            f"PR-AUC={fold_pr_auc:.6f}"
        )


    oof_results[
        set_name
    ] = {

        "y_true":
            np.array(all_y_true),

        "y_prob":
            np.array(all_y_prob),

        "fold_detail":
            pd.DataFrame(
                fold_detail
            ),

        "C":
            best_C,

        "Penalty":
            best_penalty,

        "Solver":
            best_solver
    }


OOF 수집 | Set 5
Fold 1 | Weight=168.411811 | PR-AUC=0.519429
Fold 2 | Weight=174.772875 | PR-AUC=0.560903
Fold 3 | Weight=171.036778 | PR-AUC=0.533482

OOF 수집 | Set 2
Fold 1 | Weight=168.411811 | PR-AUC=0.424218
Fold 2 | Weight=174.772875 | PR-AUC=0.471942
Fold 3 | Weight=171.036778 | PR-AUC=0.436986

OOF 수집 | Set 1
Fold 1 | Weight=168.411811 | PR-AUC=0.422112
Fold 2 | Weight=174.772875 | PR-AUC=0.470722
Fold 3 | Weight=171.036778 | PR-AUC=0.434713

OOF 수집 | Set 3
Fold 1 | Weight=168.411811 | PR-AUC=0.420746
Fold 2 | Weight=174.772875 | PR-AUC=0.469597
Fold 3 | Weight=171.036778 | PR-AUC=0.434411

OOF 수집 | Set 4
Fold 1 | Weight=168.411811 | PR-AUC=0.393398
Fold 2 | Weight=174.772875 | PR-AUC=0.441352
Fold 3 | Weight=171.036778 | PR-AUC=0.408172


In [100]:
# ==========================================
# STEP 22. Feature Set별 최적 Threshold 탐색
# ==========================================

threshold_results = []


# 더 촘촘하게 탐색
thresholds = np.arange(
    0.01,
    1.00,
    0.01
)


for set_name, result in oof_results.items():

    y_true = result[
        "y_true"
    ]

    y_prob = result[
        "y_prob"
    ]


    for threshold in thresholds:

        y_pred = (
            y_prob >= threshold
        ).astype(int)


        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )


        tn, fp, fn, tp = (
            confusion_matrix(
                y_true,
                y_pred,
                labels=[0, 1]
            )
            .ravel()
        )


        threshold_results.append({

            "Feature Set":
                set_name,

            "Threshold":
                threshold,

            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1,

            "TP":
                tp,

            "FP":
                fp,

            "FN":
                fn,

            "TN":
                tn
        })


threshold_results_df = pd.DataFrame(
    threshold_results
)


display(
    threshold_results_df.head()
)

,Feature Set,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,Set 5,0.010000,0.010077,0.997554,0.019952,3670,360543,9,284114
1,Set 5,0.020000,0.013811,0.993748,0.027244,3656,261058,23,383599
2,Set 5,0.030000,0.016376,0.990758,0.032220,3645,218933,34,425724
3,Set 5,0.040000,0.018801,0.988584,0.036900,3637,189812,42,454845
4,Set 5,0.050000,0.021351,0.986681,0.041798,3630,166384,49,478273


In [101]:
# ==========================================
# STEP 23. Feature Set별 Best Threshold
# ==========================================

best_threshold_df = (

    threshold_results_df

    .sort_values(
        [
            "Feature Set",
            "F1",
            "Recall",
            "Precision"
        ],

        ascending=[
            True,
            False,
            False,
            False
        ]
    )

    .groupby(
        "Feature Set",
        as_index=False
    )

    .first()
)


best_threshold_df = (
    best_threshold_df
    .sort_values(
        "F1",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    best_threshold_df
)

,Feature Set,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,Set 5,0.970000,0.581628,0.695298,0.633403,2558,1840,1121,642817
1,Set 4,0.980000,0.473984,0.633868,0.542389,2332,2588,1347,642069
2,Set 3,0.990000,0.497841,0.595542,0.542327,2191,2210,1488,642447
3,Set 2,0.990000,0.494046,0.597717,0.540959,2199,2252,1480,642405
4,Set 1,0.990000,0.494142,0.596086,0.540347,2193,2245,1486,642412


In [102]:
# ==========================================
# STEP 24. 최적 Threshold 최종 성능
# ==========================================

final_results = []


for _, row in best_threshold_df.iterrows():

    set_name = row[
        "Feature Set"
    ]

    best_threshold = row[
        "Threshold"
    ]


    result = oof_results[
        set_name
    ]


    y_true = result[
        "y_true"
    ]

    y_prob = result[
        "y_prob"
    ]


    y_pred = (
        y_prob >= best_threshold
    ).astype(int)


    # ------------------------------------------
    # PR-AUC
    # ------------------------------------------
    pr_auc = average_precision_score(
        y_true,
        y_prob
    )


    # ------------------------------------------
    # Precision / Recall / F1
    # ------------------------------------------
    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    # ------------------------------------------
    # Confusion Matrix
    # ------------------------------------------
    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        )
        .ravel()
    )


    final_results.append({

        "Feature Set":
            set_name,

        "C":
            result["C"],

        "Penalty":
            result["Penalty"],

        "Solver":
            result["Solver"],

        "Best Threshold":
            best_threshold,

        "PR-AUC":
            pr_auc,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "TP":
            tp,

        "FP":
            fp,

        "FN":
            fn,

        "TN":
            tn
    })


final_results_df = pd.DataFrame(
    final_results
)


final_results_df = (
    final_results_df
    .sort_values(
        [
            "PR-AUC",
            "F1"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


final_results_df.insert(
    0,
    "Rank",
    range(
        1,
        len(final_results_df) + 1
    )
)


display(
    final_results_df
)

,Rank,Feature Set,C,Penalty,Solver,Best Threshold,PR-AUC,Precision,Recall,F1,TP,FP,FN,TN
0,1,Set 5,0.010000,l2,liblinear,0.970000,0.536309,0.581628,0.695298,0.633403,2558,1840,1121,642817
1,2,Set 2,0.010000,l2,liblinear,0.990000,0.442127,0.494046,0.597717,0.540959,2199,2252,1480,642405
2,3,Set 1,0.010000,l2,liblinear,0.990000,0.440304,0.494142,0.596086,0.540347,2193,2245,1486,642412
3,4,Set 3,0.010000,l2,liblinear,0.990000,0.439444,0.497841,0.595542,0.542327,2191,2210,1488,642447
4,5,Set 4,0.010000,l2,liblinear,0.980000,0.412992,0.473984,0.633868,0.542389,2332,2588,1347,642069


In [103]:
# ==========================================
# STEP 25. 최종 Confusion Matrix 출력
# ==========================================

for _, row in final_results_df.iterrows():

    set_name = row[
        "Feature Set"
    ]

    threshold = row[
        "Best Threshold"
    ]

    y_true = oof_results[
        set_name
    ]["y_true"]

    y_prob = oof_results[
        set_name
    ]["y_prob"]

    y_pred = (
        y_prob >= threshold
    ).astype(int)


    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )


    print("\n" + "=" * 60)
    print(set_name)
    print(
        f"Best Threshold = "
        f"{threshold:.2f}"
    )
    print("=" * 60)

    print(cm)

    print(
        f"\nTN={cm[0,0]:,} | "
        f"FP={cm[0,1]:,}"
    )

    print(
        f"FN={cm[1,0]:,} | "
        f"TP={cm[1,1]:,}"
    )


Set 5
Best Threshold = 0.97
[[642817   1840]
 [  1121   2558]]

TN=642,817 | FP=1,840
FN=1,121 | TP=2,558

Set 2
Best Threshold = 0.99
[[642405   2252]
 [  1480   2199]]

TN=642,405 | FP=2,252
FN=1,480 | TP=2,199

Set 1
Best Threshold = 0.99
[[642412   2245]
 [  1486   2193]]

TN=642,412 | FP=2,245
FN=1,486 | TP=2,193

Set 3
Best Threshold = 0.99
[[642447   2210]
 [  1488   2191]]

TN=642,447 | FP=2,210
FN=1,488 | TP=2,191

Set 4
Best Threshold = 0.98
[[642069   2588]
 [  1347   2332]]

TN=642,069 | FP=2,588
FN=1,347 | TP=2,332


In [104]:
# ==========================================
# STEP 26. Logistic Regression 최종 요약
# ==========================================

logistic_summary = final_results_df.copy()

logistic_summary["Model"] = "Logistic Regression"

logistic_summary = logistic_summary[
    [
        "Model",
        "Rank",
        "Feature Set",
        "C",
        "Penalty",
        "Solver",
        "Best Threshold",
        "PR-AUC",
        "Precision",
        "Recall",
        "F1",
        "TP",
        "FP",
        "FN",
        "TN"
    ]
]

display(logistic_summary)

,Model,Rank,Feature Set,C,Penalty,Solver,Best Threshold,PR-AUC,Precision,Recall,F1,TP,FP,FN,TN
0,Logistic Regression,1,Set 5,0.010000,l2,liblinear,0.970000,0.536309,0.581628,0.695298,0.633403,2558,1840,1121,642817
1,Logistic Regression,2,Set 2,0.010000,l2,liblinear,0.990000,0.442127,0.494046,0.597717,0.540959,2199,2252,1480,642405
2,Logistic Regression,3,Set 1,0.010000,l2,liblinear,0.990000,0.440304,0.494142,0.596086,0.540347,2193,2245,1486,642412
3,Logistic Regression,4,Set 3,0.010000,l2,liblinear,0.990000,0.439444,0.497841,0.595542,0.542327,2191,2210,1488,642447
4,Logistic Regression,5,Set 4,0.010000,l2,liblinear,0.980000,0.412992,0.473984,0.633868,0.542389,2332,2588,1347,642069


In [105]:
# ==========================================
# STEP 27. 공통 Feature Set 3 성능 추출
# ==========================================

logistic_set3 = (
    final_results_df[
        final_results_df["Feature Set"] == "Set 3"
    ]
    .copy()
)

logistic_set3["Model"] = "Logistic Regression"

logistic_set3 = logistic_set3[
    [
        "Model",
        "Feature Set",
        "Best Threshold",
        "PR-AUC",
        "Precision",
        "Recall",
        "F1",
        "TP",
        "FP",
        "FN",
        "TN"
    ]
]

display(logistic_set3)

,Model,Feature Set,Best Threshold,PR-AUC,Precision,Recall,F1,TP,FP,FN,TN
3,Logistic Regression,Set 3,0.990000,0.439444,0.497841,0.595542,0.542327,2191,2210,1488,642447


## STEP 27. 공통 변수 조합(Set 3) 기준 Logistic Regression 성능

최종 알고리즘 비교에서는 각 모델마다 서로 다른 변수 조합을 사용하는 것보다,
동일한 변수 조합을 적용하여 알고리즘 자체의 성능 차이를 비교하는 것이 더 공정하다.

따라서 LightGBM 분석을 통해 최종 선정된 **Set 3**을 공통 입력 변수 조합으로 사용하고,이 결과를 Random Forest, XGBoost, LightGBM의 Set 3 결과와 비교하여
최종 모델 선정 근거로 사용한다.

In [106]:
# ==========================================
# STEP 28. 튜닝 전후 PR-AUC 개선폭 비교
# ==========================================

before_after = baseline_cv_df[
    [
        "Feature Set",
        "Mean PR-AUC"
    ]
].copy()

before_after = before_after.rename(
    columns={
        "Mean PR-AUC": "Baseline Mean PR-AUC"
    }
)

after_df = best_params_df[
    [
        "Feature Set",
        "Mean PR-AUC",
        "Std PR-AUC"
    ]
].copy()

after_df = after_df.rename(
    columns={
        "Mean PR-AUC": "Tuned Mean PR-AUC",
        "Std PR-AUC": "Tuned Std PR-AUC"
    }
)

before_after = before_after.merge(
    after_df,
    on="Feature Set",
    how="left"
)

before_after["PR-AUC Improvement"] = (
    before_after["Tuned Mean PR-AUC"]
    - before_after["Baseline Mean PR-AUC"]
)

before_after = before_after.sort_values(
    "Tuned Mean PR-AUC",
    ascending=False
).reset_index(drop=True)

display(before_after)

,Feature Set,Baseline Mean PR-AUC,Tuned Mean PR-AUC,Tuned Std PR-AUC,PR-AUC Improvement
0,Set 5,0.533285,0.537938,0.017222,0.004653
1,Set 2,0.440841,0.444382,0.020173,0.003541
2,Set 1,0.438932,0.442516,0.020597,0.003584
3,Set 3,0.437901,0.441584,0.020578,0.003683
4,Set 4,0.410402,0.414307,0.020052,0.003905


## STEP 28. 하이퍼파라미터 튜닝 전후 성능 변화

하이퍼파라미터 튜닝이 실제로 Logistic Regression의 성능을 얼마나 개선했는지 확인한다.

튜닝 전 기본 3-Fold 평균 PR-AUC와
튜닝 후 최적 설정의 평균 PR-AUC를 비교한다.

만약 튜닝 이후에도 PR-AUC 개선폭이 매우 작다면,
성능 한계가 단순한 설정값 문제라기보다
Logistic Regression의 선형 결정구조 자체와 관련될 가능성을 고려할 수 있다.

In [107]:
# ==========================================
# STEP 29. Set 3 최종 Logistic 계수 확인
# ==========================================

set_name = "Set 3"
features = feature_sets[set_name]

best_row = best_params_df[
    best_params_df["Feature Set"] == set_name
].iloc[0]

# 마지막 Fold의 Train 데이터를 사용
fold_name = "Fold 3"
fold_info = folds[fold_name]

fold_train, fold_val = get_fold_data(
    segments,
    fold_info
)

X_train = fold_train[features]
y_train = fold_train["is_fraud"]

normal = (y_train == 0).sum()
fraud = (y_train == 1).sum()

scale_pos_weight = normal / fraud

class_weight = {
    0: 1.0,
    1: scale_pos_weight
}

model = make_logistic_pipeline(
    features=features,
    class_weight=class_weight,
    C=best_row["C"],
    penalty=best_row["Penalty"],
    solver=best_row["Solver"]
)

model.fit(
    X_train,
    y_train
)

preprocessor = model.named_steps["preprocessor"]
logistic = model.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": logistic.coef_[0]
})

coef_df["Abs Coefficient"] = (
    coef_df["Coefficient"].abs()
)

coef_df = coef_df.sort_values(
    "Abs Coefficient",
    ascending=False
).reset_index(drop=True)

display(coef_df)

,Feature,Coefficient,Abs Coefficient
0,cat__category_gas_transport,2.480543,2.480543
1,cat__category_grocery_net,2.384057,2.384057
2,cat__category_grocery_pos,1.899996,1.899996
3,cat__category_shopping_pos,-1.758404,1.758404
4,cat__category_shopping_net,-1.569633,1.569633
5,cat__category_personal_care,1.345756,1.345756
6,cat__category_kids_pets,1.171553,1.171553
7,cat__category_misc_pos,1.091308,1.091308
8,num__amt,0.993082,0.993082
9,cat__category_food_dining,0.863367,0.863367


## STEP 29. Logistic Regression 계수 확인

Logistic Regression은 각 변수에 하나의 계수를 부여하고,
이 계수들의 선형 결합을 이용해 이상거래 확률을 계산한다.

- 계수가 양수이면 해당 값이 증가할수록 이상거래로 판단할 가능성이 커진다.
- 계수가 음수이면 해당 값이 증가할수록 정상거래로 판단할 가능성이 커진다.
- 절댓값이 클수록 모델의 판단에 상대적으로 큰 영향을 준다.

다만 Logistic Regression은 기본적으로 변수와 이상거래 여부 사이의
**선형적인 관계**를 학습하는 모델이다.

따라서 여러 변수의 복잡한 상호작용이나
특정 구간에서만 나타나는 비선형 패턴을 직접 학습하는 데에는 한계가 있다.

# Logistic Regression 성능 해석 및 모델 선정 결과 (피드백 반영)

## 1. Logistic Regression의 성능이 상대적으로 낮게 나타난 이유

본 프로젝트에서 Logistic Regression은 이상거래 탐지를 위한
**Baseline 모델**로 사용하였다.

Logistic Regression은 계산 속도가 빠르고 각 변수의 영향을 명확하게
해석할 수 있다는 장점이 있지만, 입력 변수들을 기본적으로
**선형 결합(Linear Combination)**하여 이상거래 확률을 계산한다.

즉, 모델은 다음과 같은 구조로 거래의 위험도를 계산한다.

`각 변수 × 해당 변수의 계수 → 모두 더함 → 이상거래 확률 계산`

이러한 구조는 변수와 이상거래 사이의 관계가 비교적 단순할 때 효과적이지만,
본 프로젝트의 이상거래 데이터처럼 여러 조건이 복합적으로 작용하는 경우에는
성능에 한계가 나타날 수 있다.


### 1-1. 이상거래 패턴에는 비선형 관계가 존재할 가능성이 있음

EDA 결과 거래금액과 이상거래 사이에는 단순히

> "거래금액이 증가할수록 이상거래 위험도도 계속 증가한다."

와 같은 관계만 존재하지 않았다.

실제로 거래금액 분석에서는 특정 금액 구간에서 이상거래가 집중되는 등
**비선형(Non-linear) 관계가 존재할 가능성**이 확인되었다.

따라서 이상거래 여부는 단순한 직선 형태의 관계보다는
특정 조건이나 구간에 따라 위험도가 크게 달라지는 문제라고 볼 수 있다.

Logistic Regression은 이러한 복잡한 비선형 관계를
기본 구조만으로 직접 학습하는 데 한계가 있다.


### 1-2. 여러 변수의 상호작용을 직접 학습하기 어려움

이상거래는 하나의 변수만으로 결정되지 않는다.

예를 들어 다음과 같은 상황을 생각할 수 있다.

- 거래금액이 평소보다 매우 높고
- 짧은 시간 동안 여러 번 거래가 발생했으며
- 평소와 다른 시간대에 거래가 발생한 경우

각 조건을 개별적으로 보면 반드시 이상거래라고 판단하기 어렵지만,
**여러 조건이 동시에 발생하면 이상거래 위험이 크게 증가할 수 있다.**

Logistic Regression은 각 변수에 하나의 계수를 부여하여
이들을 선형적으로 결합하기 때문에 이러한 복잡한 변수 간 상호작용을
자동으로 찾아내는 데 제한이 있다.

반면 LightGBM, XGBoost, Random Forest와 같은 Tree-based 모델은

`특정 변수의 조건 → 다른 변수의 조건 → 또 다른 변수의 조건`

과 같은 분기 구조를 통해 복잡한 비선형 관계와 변수 간 상호작용을
보다 유연하게 학습할 수 있다.


### 1-3. 매우 심한 클래스 불균형 문제

본 데이터에서는 정상거래가 약 99.42%,
이상거래가 약 0.58%를 차지할 정도로 클래스 불균형이 매우 크다.

따라서 단순히 대부분의 거래를 정상으로 분류하는 것만으로도
Accuracy는 매우 높게 나타날 수 있다.

본 프로젝트에서는 이러한 문제를 완화하기 위해 각 Fold의 Train 데이터에서

`정상거래 수 ÷ 이상거래 수`

를 계산하여 이상거래에 더 높은 class weight를 부여하였다.

그러나 클래스 가중치는 이상거래의 중요도를 높여주는 방법이지,
Logistic Regression 자체를 비선형 모델로 바꾸는 것은 아니다.

따라서 불균형 문제를 보정하더라도
복잡한 이상거래 패턴을 표현하는 모델 구조 자체의 한계는 남아 있다.


### 1-4. 하이퍼파라미터 튜닝만으로 구조적 한계를 해결하기 어려움

본 실험에서는 Logistic Regression의 `C`, `penalty`, `solver`를
변경하면서 하이퍼파라미터 튜닝을 수행하였다.

하지만 이러한 하이퍼파라미터는 주로 규제 강도와 최적화 방법을 조정하는 것으로,
Logistic Regression의 기본적인 선형 결정구조 자체를 변경하지는 않는다.

따라서 튜닝을 통해 성능을 일정 부분 개선할 수는 있지만,
데이터에 존재하는 복잡한 비선형 관계까지 자동으로 학습할 수 있게 되는 것은 아니다.

결과적으로 본 데이터에서는 Tree-based 모델이 복잡한 이상거래 패턴을
보다 효과적으로 학습하면서 PR-AUC, Precision, F1-score 등의 지표에서
Logistic Regression보다 높은 성능을 기록한 것으로 해석할 수 있다.


---

# 2. 그렇다면 Logistic Regression은 필요 없는 모델인가?

그렇지 않다.

Logistic Regression은 최종 예측 성능에서는 다른 복잡한 모델보다
불리할 수 있지만, **높은 해석 가능성(Interpretability)**이라는
매우 중요한 장점을 가지고 있다.

특히 각 변수에 대한 회귀계수(Coefficient)를 직접 확인할 수 있기 때문에,

> "어떤 변수가 이상거래 판단에 어느 방향으로 영향을 주었는가?"

를 비교적 명확하게 설명할 수 있다.

따라서 본 프로젝트에서 Logistic Regression은 단순히 성능 경쟁을 위한 모델이 아니라,
복잡한 모델을 사용할 필요가 있는지를 판단하기 위한 **설명 가능한 Baseline 모델**이라는
의미를 가진다.


---

# 3. 회귀계수를 이용한 해석 예시

STEP 29에서는 최종 Set 3 Logistic Regression의 회귀계수를 확인하였다.

그중 `amt`의 결과는 다음과 같다.

| Feature | Coefficient |
|---|---:|
| `amt` | **+0.993082** |

`amt`는 거래금액을 의미한다.

본 모델에서는 수치형 변수를 StandardScaler로 표준화한 후 Logistic Regression에
입력하였으며, `amt`의 회귀계수는 **+0.993082**로 나타났다.

양수의 회귀계수는 다른 조건이 동일하다고 가정할 때 해당 변수 값이 증가할수록
모델이 이상거래로 판단하는 방향으로 위험점수가 증가한다는 것을 의미한다.

따라서 이 결과는 다음과 같이 해석할 수 있다.

> **다른 변수들이 동일한 조건에서 거래금액(amt)이 높아질수록
> Logistic Regression은 해당 거래를 이상거래로 판단하는 방향으로
> 더 높은 위험점수를 부여하였다.**

이처럼 Logistic Regression에서는 단순히

> "이 변수가 중요하다."

라고 말하는 것에서 그치지 않고,

> "이 변수의 값이 커질 때 이상거래 위험을 높이는 방향인지,
> 낮추는 방향인지"

까지 확인할 수 있다.

이는 Logistic Regression의 대표적인 장점인 **해석 가능성**을 보여준다.


---

## 4. 회귀계수 해석 시 주의점

다만 회귀계수를 인과관계로 해석해서는 안 된다.

`amt`의 계수가 양수라고 해서

> "거래금액 증가가 이상거래를 발생시킨다."

라는 의미는 아니다.

회귀계수는 어디까지나 **현재 모델이 주어진 데이터에서 학습한 예측 관계의 방향**을
보여주는 값이다.

또한 본 모델의 수치형 변수에는 StandardScaler가 적용되어 있으므로,
`amt`의 계수 0.993082를 단순히

> "거래금액이 1원 증가하면 위험도가 0.993 증가한다."

라고 해석해서도 안 된다.

표준화된 입력값을 기준으로 계산된 계수이므로
주로 변수의 영향 방향과 모델의 판단 구조를 설명하는 데 활용하는 것이 적절하다.


---

# 5. 최종 해석

Logistic Regression은 각 변수의 영향을 선형적으로 결합하기 때문에
복잡한 비선형 패턴과 변수 간 상호작용이 존재하는 이상거래 탐지 문제에서는
Tree-based 모델보다 낮은 성능을 보일 수 있다.

특히 본 프로젝트에서는 거래금액, 고객의 평소 소비 패턴,
단기간 거래 빈도, 거래 시간 등의 여러 정보가 복합적으로 작용하므로
단순한 선형 결정구조만으로 모든 이상거래 패턴을 표현하기에는 한계가 있었다.

반면 Logistic Regression은 회귀계수를 통해
각 변수가 모델의 이상거래 판단에 미치는 방향을 직접 확인할 수 있다는
명확한 장점을 가진다.

실제로 Set 3의 `amt`는 **+0.993082**의 계수를 보여,
다른 조건이 동일할 때 거래금액이 증가할수록 이상거래 위험점수를 높이는 방향으로
모델이 학습했음을 확인할 수 있었다.

따라서 본 프로젝트에서 Logistic Regression은
**최종 성능이 가장 높은 모델이라기보다, 데이터의 선형 관계를 확인하고
복잡한 모델의 필요성을 판단할 수 있게 해주는 해석 가능한 Baseline 모델**로서
의미가 있다고 판단하였다.